# To be able to edit and use this Notebook:

0. Learn about how to use google colaboratory [video]()
1. in the file menu (top left), click ```open in playground```
3. still in the file menu, click ```save copy in drive```, to make your own personalized and editable copy of this file.
4. edit as you like. If something breaks irreparably, either:
  1. restart the ```Runtime```
  2. or go back to step 1.


# Dynamics of the Adaptive Exponential Integrate and Fire

In the last project we dove into the equations of the Hodgkin Huxley model of spike generation. That model is described by the temporal evolution of four state variables (quiz yourself: do you know which are the state variables of the HH?), and hence its dynamics are quite complex to analyze. 

In this project we will learn how to simulate and parameterize the **Adaptive Exponential Integrate and Fire model** (AdEx).  You will observe how the parameters of the AdEx model can lead it to produce a vast variety of spiking patterns. We will review type 1 and type 2 neuronal firing (introduced in the lecture) and will calculate **I x F** curves (i.e., injected current **I** vs spiking frequency  **F**) of the different types, and use the IxF curves to distinguish neuronal behavior.


# Learning Objectives

**After this project you will be able to:**
- Recognize the state variables of the AdEx model.
- Implement an AdEx model in Brian and perform simulations with different parameters.
- Visualize the role of different parameters of the AdEx model on the dynamics of spiking.
- Keep your neurons unit-consistent (SI units: Ampere, Volts, Farads, Siemens).
- Calculate and display F x I curves and use them to compare different neuronal models.
- Use F x I curves to distinguish between 'integrators (type 1) and 'resonators' (type 2).


# Terminology

- **Dynamical system**: A mathematical description of the rules governing the state evolution of a system.
- **State variable**: one of the variables that is needed to describe the current state of a system.
- **Phase space**: The space occupied by the all the state variables of a given system.
- **Parameter**: A property of a sytem that influences its dynamics, usually a variable that changes slower than the state variables and is not directly influenced by them.
- **F x I curves**: Curves that measure the relationship between spiking rate  (Frequency) and injected current (I).
- **Equilibria**: Points in phase space where the state maps onto itself.
- **Nullclines**: Curves where the gradient for one particular state variable is zero (dx/dt = 0).
- **Resonators and Integrators** (Type II and I): Descriptions of neuronal types as a function of their spiking properties.

## Initialization

## Run This First

Run the next code cell before the rest of the notebook.

- In Google Colab it installs any missing notebook-only packages and enables widget support.
- In local JupyterLab it only verifies imports against your active environment.
- Local setup: create a virtual environment and install the packages in `requirements-notebooks.txt`.


In [ ]:
# Notebook runtime setup for Google Colab and local JupyterLab.
import importlib
import subprocess
import sys

try:
    from google.colab import output as colab_output
    IS_COLAB = True
except ImportError:
    colab_output = None
    IS_COLAB = False


def ensure_notebook_packages(requirements):
    if not IS_COLAB:
        return

    missing = []
    for package_name, module_name in requirements:
        if importlib.util.find_spec(module_name) is None:
            missing.append(package_name)

    if missing:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])


NOTEBOOK_REQUIREMENTS = [('brian2', 'brian2'), ('brian2tools', 'brian2tools')]
ensure_notebook_packages(NOTEBOOK_REQUIREMENTS)

if IS_COLAB:
    colab_output.enable_custom_widget_manager()


In [ ]:
from brian2 import *
import time
from scipy import optimize
import numpy as np
from scipy.integrate import odeint
import matplotlib.pyplot as plt
from sympy.solvers import solve
from sympy import symbols
import sympy as sp
%matplotlib inline

cl=Clock(dt=0.1*ms)


## The AdEx Integrate and Fire Model

The Hodkgin-Huxley model, in an attempt to capture much physiological detail has a multitude of equations. That is both computationally expensive and cumbersome. Mathematical work to understand the essential aspects of the HH dynamics have led to significant simplifications, capable of reproducing a nearly complete spectrum of spiking phenomena including tonic and phasic spiking, bursting, chattering, rebound spikes, and more. Simplified equations are more amenable to analysis and computation and may retain the spike generation behaviors of the more complex model, at a very moderate cost. In fact simplified models can reproduce more complex spiking phenomenology than the more complex models such as the Hodgkin-Huxley or the Goldman-Hodgkin-Katz models.

 In modern computational neuroscience one of the most used simplified models is the **Adaptive Exponential** neuronal model, or *AdEx* for short. It is of the 'integrate and fire' type, because unlike the HH model where all the state variables are continuous. It has a 'threshold value', which produces a spike and resets to a given reset potential. Its dynamics are defined by **two state variables**, the membrane potential and a *refractory* variable. We call it a **2D Integrate and Fire** model. The latter indicating the combined state of the refractory mechanisms, which repolarize the neuron (such as sodium inactivation and potassium activation). In the AdEx model, the positive feedback of Sodium channels that lead to the spike is replaced by an exponential function with a fast rise, able to correctly reproduce the spike shapes of many types of cortical neurons.




Below we present the AdEx equations, with parameters and state variables.

For the dynamics of the **membrane potential** we have the following variables, common base units and parameters:

> $V_m$ - Membrane potential, a __state variable__ (milivolts)  \\
> $g_L$ - Leak conductance, a __parameter__ passive decay through the membrane (siemems) \\
> $E_L$ - The reversal potential of the passive leak, a __parameter__ (volts) \\
> $V_T$ - The spike 'threshold' (volts) \\
>$\Delta_T$ - A 'slope' factor, a __parameter__ controling the rise time of the spike (ms) \\
> $I$ - Injected current, a __parameter__ either entering synaptically or through a current clamp (Amperes). \\
> $C$ - Capacitance, a __parameter__ determines how fast the membrane potential changes \\
> $w$ - Refractory current (Ampere), the other __state variable__. Notice that it has the opposite sign of $I$.

For the dynamics of the **refractory variable** $w$ representing the potassium currents and inactivations, in addition to some of the above, we have:

> $\tau_w$ - adaptation time constant (Ampere) \\
> $a$ - subthreshold adaptation \\
> $b$ - spike triggered adaptation \\




**Membrane Dynamics**


\begin{align*}\frac{\mathrm{d}V_m}{\mathrm{d}t} &= \frac{ \overbrace{g_L \Delta_T e^{\frac{- V_T + Vm}{\Delta_T}}}^{spike \ rise} + \overbrace{g_L \left(E_L - V_m\right)}^{passive \ component} - w + I}{C} && \text{(unit of $V_m$: $\mathrm{Volt}$)}\\ \frac{\mathrm{d}w}{\mathrm{d}t} &= \frac{a \left(- E_L + V_m\right) - w}{\tau_w} && \text{(unit of $w$: $\mathrm{Ampere}$)}\end{align*} 


**On spiking:**

- Threshold condition:

$$V_m>V_{cut}$$

- Reset statements:

$$  
 \{ \begin{array}{ll}
  V_m = V_r \\
  w  \  +=b \\
 \end{array}
$$


As you will notice in the equations above, the AdEx has a number of parameters (gL, EL, VT, a, tauw, Vcut, Vr, b). It also has two **state variables**, $V_m$ and $w$. 

Different combinations of values for the **parameters** will lead to different types of neuronal responses, including spike frequency, spike shape and response to input. In other words, these parameters will influence the dynamics of the neuronal model. By selecting the parameters judiciously, one can reproduce a variety of spike forms. These parameter sets can then be analyzed, and give insight into the physiology and spike generation mechanisms of real neurons.

### Warm up: Write the AdEx equations

In order to create the AdEx model, we start by defining the differential equations for membrane dynamics in variable ```state_eqs```. 

We write the AdEx equations below, in a format that Brian understands. Remember to add the 'base units', ```volt``` and ```amp```.

In [ ]:
# your code
state_eqs = ''' 
dvm/dt = (gL*(EL-vm)+gL*DealtaT*exp((vm-VT/DeltaT)+I_input-w)/C : volt
dw/dt= (a*(vm-EL)- w)/tauw : amp
 '''


#### Our Solution

In [ ]:
state_eqs='''
dvm/dt = (gL*(EL-vm)+gL*DeltaT*exp((vm-VT)/DeltaT)+I_input-w)/C : volt
dw/dt = (a*(vm-EL)-w)/tauw : amp
'''


Additionally, we can define conditions for the spiking model. These can be passed on to the model later when we define NeuronGroup, as seperate parameters. The conditions of interest are about spiking threshold ```threshold_eq``` and resetting the model after a spike ```reset_eqs```. The equations for both parameters can be found above.

We also need to define the unit of the input current ```I```, ```amp```, and we add those to the variable ```input_eqs```.


In [ ]:
# to keep unit consistency we must  define the unit of I ('input', the applied current)
input_eqs='''
I_input : amp
'''

# we spike if vm is larger than Vcut 
threshold_eq='vm>Vcut'

# after a spike we reset to Vr and add 'b' to w.
reset_eqs="vm=Vr; w+=b"


# Parameterizing Models

## A function that takes a neuron and returns a parameterized neuron

--- 
#### Dictionaries for Parameters

In the code below we define some default values for neurons parameters and an experiment, and create a **function** that returns the parameters of a given neuron type in a **dictionary**. This dictionary can then be passed to the ```neuronGroup()``` object, via it's **namespace**, which contains all the runtime parameters of the model we will run.

The advantage of using dictionaries for parameterizations are many:
1. orderliness: we keep all parameter sets in one place;
2. cleanliness: we can change the parameters of the neuron at any point in the code below without having to copy the values;
3. flexibility: we can easily change all the parameters at once;
4. clarity: we can easily access all user variables at once via the namespace;

---

We wrote the function below for your convenience. The function ```set_parameters(name)``` takes an initialized ```neuron``` from Brian and loads it up with ```parameters``` specified by the neuron type we want (tonic, bursting and so on). You can think of this function as differentiating a stem cell ```neuron``` into a specific neuron.

**Task: Read the function below. Try to spot which are the 'default parameters' and which are the parameters that are set when you select a particular neuronal type.**

_Important: in the function below we are not testing if the parameters into the function are meaningful or correct (for example, if 'neuron' is indeed a Brian's neuron.). These kinds of checks are useful in larger coding projects. One could test if the parameters are sensible and only execute the function if they are, otherwise returning an error or warning._

#### A python function to ```set_parameters(name)```

In [ ]:
def set_parameters(name):
  # stimulation parameters:
  I0=0*nA
  I1 = 1.0*nA
  dv0=0*mV # voltage transient
  T0=50*ms
  T1=1000*ms
  T2=200*ms

  # variable parameters:
  if name=='phasic':
      tauw=150*ms
      a=2*C/tauw # type II
      b=0*nA
      Vr=-70.6*mV
      gL=30*nS
      EL=-60*mV
      VT=-50.4*mV
      I_john=(1+a/gL)*log(1+taum/tauw)-(1+taum/tauw)
      I0=gL*DeltaT*I_john+(VT-EL)*(gL+a)-0.03*nA
      I=I0
      T0=100*ms
      T1=200*ms
      T2=20*ms
      dv0=2.5*mV
  elif name=='regular':
      tauw=144*ms
      a=4*nS
      b=0.0805*nA
      Vr=-70.6*mV
      I=1*nA
  elif name=='fast':
      # Type II: a > C/tauw
      tauw=144*ms
      a=2*C/tauw
      b=0*nA
      Vr=-70.6*mV
      EL=-70.6*mV # Same as changing I
      VT=-50.4*mV
      gL=30*nS
      I_john=(1+a/gL)*log(1+taum/tauw)-(1+taum/tauw)
      I=gL*DeltaT*I_john+(VT-EL)*(gL+a)+0.01*nA
      I0=I-0.1*nA
      T0=200*ms
  elif name=='bursting_tonic':
      tauw=20*ms
      a=4*nS
      b=0.5*nA
      VT=-50.4*mV
      Vr=VT+5*mV
      I=.8*nA

  # Create namespace:
  namespace_params = {
      'I0':I0,
      'I1':I1,
      'dv0':dv0,
      'T0':T0,
      'T1':T1,
      'T2':T2,
  
      'tauw':tauw,
      'a':a,
      'b':b,
      'Vr':Vr,
      'I':I}
      
  return namespace_params 


### Example:  Parameterize a Neuron
In the field below, we create a neuron and parameterize it as a *regularly spiking* neuron ('regular' from the above function).

In [ ]:
start_scope()

# general parameters:
C=281*pF # Can be fixed
gL=30*nS
taum=C/gL
EL=-70.6*mV # Same as changing I
VT=-50.4*mV
DeltaT=2*mV
Vcut=VT+5*DeltaT

# set the parameters
namespace_params = set_parameters('bursting_tonic') #Return parameters, applied current, and inital vm

# create a neuron groupp containing 1 neuron defined by equations above
neuron = NeuronGroup(1,model=state_eqs+input_eqs,threshold=threshold_eq, reset=reset_eqs, namespace=namespace_params, method='euler',name='neuron')

# check the parameters of your differentiated neuron
print(neuron.namespace)


### Record Neuronal Behavior

A ```StateMonitor()``` is a function that 'records' the selected output variables of the neurons. We create StateMonitors to be able to observe the outputs of the neuron. 

**Note:** In general, simulators do not save every variable computed, as this can lead to enormous memory requirements.

In [ ]:
# to retrieve the values of vm of the first neuron, write 'mon_v.vm[0]
mon = StateMonitor(neuron,['vm', 'I_input', 'w'],record=0) 

# the spike monitor records the times of the spikes in the array 'spikes.t'
spikes=SpikeMonitor(neuron)


### Run the Simulation (according to parameters)

Note: after parameterizing the neuron also contains the parameters necessary to run the simulation. E.g., I0 is the initial current applied, T0 is the duration for I0.

In [ ]:
# Set Initial vm and I accordign to the parameters saved in our 'neuron'
neuron.vm = neuron.namespace['dv0']
neuron.I_input = neuron.namespace['I0']

# initial duration (relaxation)
run(neuron.namespace['T0'])

# apply current and run for duration T1
neuron.I_input = neuron.namespace['I1']
run(neuron.namespace['T1'])

# back to no current and relax
neuron.vm = neuron.namespace['dv0']
neuron.I_input = neuron.namespace['I0']
run(50*ms)

# we run the neuron for T2 seconds
run(neuron.namespace['T2'])


## Plot the results

Here we plot the results of our simulation. Note that to successfully pass the recorded variables to the plot functions, we need to remove the units from our monitors. We can do this by dividing the variable array by the appropriate unit. 

In [ ]:
# Plot the neuron's membrane potential
figure(figsize=(12,4))
plot(mon.t/ms,mon.vm[0]/mV)
ylabel('V (mV)')
xlabel('t (ms)')


### Example: Plot all the state variables in separate subplots, including:
1. Membrane potential trace
2. Refractory variable trace
3. Current injection
4. State space (membrane potential vs refractory variable)

#### Our Solution

In [ ]:
start_scope()

# general parameters:
C=281*pF # Can be fixed
gL=30*nS
taum=C/gL
EL=-70.6*mV # Same as changing I
VT=-50.4*mV
DeltaT=2*mV
Vcut=VT+5*DeltaT

# set the parameters
namespace_params = set_parameters('regular') #Return parameters, applied current, and inital vm

# create a neuron groupp containing 1 neuron defined by equations above
neuron = NeuronGroup(1,model=state_eqs+input_eqs,threshold=threshold_eq, reset=reset_eqs, namespace=namespace_params, method='euler',name='neuron')

# check the parameters of your differentiated neuron
print(neuron.namespace)

# to retrieve the values of vm of the first neuron, write 'mon_v.vm[0]
mon = StateMonitor(neuron,['vm', 'I_input', 'w'],record=0) 

# the spike monitor records the times of the spikes in the array 'spikes.t'
spikes=SpikeMonitor(neuron)

# Set Initial vm and I accordign to the parameters saved in our 'neuron'
neuron.vm = neuron.namespace['dv0']
neuron.I_input = neuron.namespace['I0']

# initial duration (relaxation)
run(neuron.namespace['T0'])

# apply current and run for duration T1
neuron.I_input = neuron.namespace['I1']
run(neuron.namespace['T1'])

# back to no current and relax
neuron.vm = neuron.namespace['dv0']
neuron.I_input = neuron.namespace['I0']
run(50*ms)

# we run the neuron for T2 seconds
run(neuron.namespace['T2'])

# Plot the neuron's membrane potential
figure(figsize=(16,10))

plt.subplot(2,2,1)
plot(mon.t/ms,mon.vm[0]/mV)
ylabel('V (mV)')
xlabel('t (ms)')

plt.subplot(2,2,2)
plot(mon.t/ms,mon.w[0]/nA)
ylabel('w (nA)')
xlabel('t (ms)')

plt.subplot(2,2,3)
plot(mon.t/ms, mon.I_input[0]/nA)
ylabel('I_input (nA)')
xlabel('t (ms)')

plt.subplot(2,2,4)
plot(mon.vm[0]/mV,mon.w[0]/nA)
ylabel('w (nA)')
xlabel('V (mV)')


# Exercises:


## Exercise 1.
Go back to the line that parameterizes the neuron ```set_parameters(neuron, type)```, choose yourself a neuron type by its name and re-run the simulation. Can you see why the neuron has the name it has? Bonus: Attempt to relate the parameters with the behavior the neuron model displays.

## Exercise 2. 

Create F x I curves for the following neuronal types: **regular**, and **bursting tonic**. Display the FxI curves in the same axes and add legends. Determine which of them is of "Type I" and which one is "Type II" (http://www.scholarpedia.org/article/Adaptive_exponential_integrate-and-fire_model#Type_I_and_Type_II_in_frequency-current_curves).

The idea is that you select a range for input currents (say from 0*nA to 1.5*nA, in 40 steps)

You can check how to calculate and display F x I curves at https://brian2.readthedocs.io/en/stable/examples/IF_curve_LIF.html

Brain's way to do it is to **sweep over the parameter space** of currents (I)via a group of unconnected neurons, where each is given a current pulse of increasing amplitude. 

Note: To compute the firing rate, be sure to only take the spikes that are due to the period of stimulation.

#### Your Code

In [ ]:
start_scope()

# general parameters:
C=281*pF # Can be fixed
gL=30*nS
taum=C/gL
EL=-70.6*mV # Same as changing I
VT=-50.4*mV
DeltaT=2*mV
Vcut=VT+5*DeltaT

# set the parameters
namespace_params = set_parameters('fast','regular') #Return parameters, applied current, and inital vm

# create a neuron groupp containing 1 neuron defined by equations above
neuron = NeuronGroup(1,model=state_eqs+input_eqs,threshold=threshold_eq, reset=reset_eqs, namespace=namespace_params, method='euler',name='neuron')

# check the parameters of your differentiated neuron
print(neuron.namespace)

# to retrieve the values of vm of the first neuron, write 'mon_v.vm[0]
mon = StateMonitor(neuron,['vm', 'I_input', 'w'],record=0) 

# the spike monitor records the times of the spikes in the array 'spikes.t'
spikes=SpikeMonitor(neuron)

# Set Initial vm and I accordign to the parameters saved in our 'neuron'
neuron.vm = neuron.namespace['dv0']
neuron.I_input = neuron.namespace['I0']

# initial duration (relaxation)
run(neuron.namespace['T0'])

# apply current and run for duration T1
neuron.I_input = neuron.namespace['I1']
run(neuron.namespace['T1'])

# back to no current and relax
neuron.vm = neuron.namespace['dv0']
neuron.I_input = neuron.namespace['I0']
run(50*ms)

# we run the neuron for T2 seconds
run(neuron.namespace['T2'])

# Plot the neuron's membrane potential
figure(figsize=(12,4))
plot(mon.t/ms,mon.vm[0]/mV)
ylabel('V (mV)')
xlabel('t (ms)')


In [ ]:
# Set the range of input currents
neuron.I_input = ''
neuron_fast.I_input = ''

# Create the spikemonitor
monitor = SpikeMonitor(neuron)
monitor_fast = SpikeMonitor(neuron_fast)

# Run the simulations
run(duration)


In [ ]:
# Plot the FxI curves
figure(figsize=(15,7))
plt.subplot(1,2,1)
plot(neuron.I_input/nA, monitor.count / duration)
xlabel('I (nA)')
ylabel('Firing rate (sp/s)')

plt.subplot(1,2,2)
plot(neuron_fast.I_input/nA, monitor_fast.count / duration)
xlabel('I (nA)')
ylabel('Firing rate (sp/s)')
show()


#### Our Solution

In [ ]:
start_scope()

# general parameters:
C=281*pF # Can be fixed
gL=30*nS
taum=C/gL
EL=-70.6*mV # Same as changing I
VT=-50.4*mV
DeltaT=2*mV
Vcut=VT+5*DeltaT

# Choose number of steps and simulation duration
n = 1000
duration = 0.5*second

# set the parameters and create a neuron groupp for the regular scenario
namespace_params = set_parameters('regular') #Return parameters, applied current, and inital vm
neuron = NeuronGroup(n,model=state_eqs+input_eqs,threshold=threshold_eq, reset=reset_eqs, namespace=namespace_params, method='euler',name='neuron')

# set the parameters and create a neuron groupp for the fast scenario
namespace_params = set_parameters('fast') #Return parameters, applied current, and inital vm
neuron_fast = NeuronGroup(n,model=state_eqs+input_eqs,threshold=threshold_eq, reset=reset_eqs, namespace=namespace_params, method='euler',name='neuron_fast')


In [ ]:
# Set the range of input currents
eqs_1 = '2.0*nA * i / (n-1)'

neuron.I_input = eqs_1
neuron_fast.I_input = eqs_1

# Create the spikemonitor
monitor = SpikeMonitor(neuron)
monitor_fast = SpikeMonitor(neuron_fast)

# Run the simulations
run(duration)


In [ ]:
from brian2tools import *

# Plot the FxI curves
figure(figsize=(15,14))
plt.subplot(2,2,1)
brian_plot(monitor)

plt.subplot(2,2,2)
brian_plot(monitor_fast)

plt.subplot(2,2,3)
plot(neuron.I_input/nA, monitor.count / duration)
xlabel('I (nA)')
ylabel('Firing rate (sp/s)')

plt.subplot(2,2,4)
plot(neuron_fast.I_input/nA, monitor_fast.count / duration)
xlabel('I (nA)')
ylabel('Firing rate (sp/s)')
show()


## References
[0] [Dimensionality Reduction and Phase Plane Analysis (ebook)](https://neuronaldynamics.epfl.ch/online/Ch4.html)

[1] [Adaptive Exponential Integrate and Fire model (Scholarpedia)](http://www.scholarpedia.org/article/Adaptive_exponential_integrate-and-fire_model)

[2]	J. Touboul and R. Brette, “Dynamics and bifurcations of the adaptive exponential integrate-and-fire model.,” Biol Cybern, vol. 99, no. 4, pp. 319–334, Nov. 2008.

[3] E. Izhikevich. "Dynamical Systems in Neuroscience". MIT press. 2017.

[4] [M. Stimberg, R. Brette, D. Goodman. "Brian 2, an intuitive and efficient neural simulator" eLife, 2019.](https://elifesciences.org/articles/47314)

# To Know More:

To deepen your understanding of mathematics behind the neurodynamics of simplified neuron models you can watch these episodes of Wulfram Gerstners' MOOC. Thereafter you should be equipped to go through the "advanced neurodynamics" addendum to this project.

- Week 4 from [Wulfram Gerstner's MOOC](https://lcnwww.epfl.ch/gerstner/NeuronalDynamics-MOOCall.html)

> The section below is provided for reference on more advanced concepts. It can be safely skipped over!

# ADVANCED NEURODYNAMICS:

This section is for advanced students with a solid calculus foundation. If you want to know more about how equations lead to diverse types of spike generation, you can use the code snippets below to analyze the dynamics of the different models in detail.

 ## Examining the State Space and Fixed Point Stability

 The code below performs in depth analysis of the stability of 2D systems.

## Find the nullcline equations

nullclines: the curves in phase space where the gradient is zero. In a 2D system with state variables V and W, the nullclines are given by two equations $\dot{V} = F(V,W) = 0$ and $\dot{W} = G(V,W) = 0$.

In [ ]:
### Define the symbos you will use to solve the equations
vm1,w1,C1,gL1,taum1,EL1,VT1,DeltaT1,Vcut1,tauw1,a1,b1,I_input1 = symbols('vm w C gL taum EL VT DeltaT Vcut tauw a b I_input')
# Use solve
vm_eq = solve((gL1*(EL1-vm1)+gL1*DeltaT1*sp.exp((vm1-VT1)/DeltaT1)+I_input1-w1)/C1,w1)
w_eq = solve((a1*(vm1-EL1)-w1)/tauw1,w1)
  
y_vm = vm_eq[0]
y_w = w_eq[0]

### Equations 
print('\n Symbolic Equations:\n','\nVm Nullcline:\n', " w = ", y_vm,'\nW Nullcline:\n', " w = ", y_w)


In [ ]:
### Substitute values of the constants to simplify the equations. 
def nl(gL,EL,VT,DeltaT,a,I):
    yvm = y_vm.subs(gL1,gL/nsiemens)
    yvm = yvm.subs(EL1,EL/mvolt)
    yvm = yvm.subs(VT1,VT/mvolt)
    yvm = yvm.subs(DeltaT1,DeltaT/mvolt)
    yvm = yvm.subs(I_input1, I/namp)
    yw = y_w.subs(a1,a/nsiemens)
    yw = yw.subs(EL1,EL/mvolt)
    return [yvm, yw]
  

eqs_nl = nl(gL,EL,VT,DeltaT,neuron.namespace['a'],neuron.namespace['I1'])
vm_0 = eqs_nl[0]
w_0 = eqs_nl[1]

### Simplified Equations for plotting
print('\n Simplified Equations:\n','\nVm Nullcline:\n', " w = ", vm_0,'\nW Nullcline:\n', " w = ", w_0)


## Find the intersection points

At the intersection of nullclines we have fixed points, which may be attractors, repellors or saddle nodes.

### Write the equations of the nullclines from your results

In [ ]:
### Choose range for vm and w  
min_lin = -100.0 #Min value for vm
max_lin = 20.0 #Max value for vm
resol = 120000 #Resolution for the linspace. Choose the number of points in your linspace. In this case we need a high resolution to find the intersections. 
vm = np.linspace(min_lin, max_lin, resol)

### Get a list with the solution to the equations. Substitute values of vm in the equations vm_0 and w_0. 
y1 = []
for i in range(len(vm)):
  y1.append(vm_0.subs(vm1,vm[i]))
y2 = []
for i in range(len(vm)):
  y2.append(w_0.subs(vm1,vm[i]))


### Find Intersections

In [ ]:
### Find the index where y1 and y2 have a difference of 0. In other words, y1-y2 = 0.
idx=np.argwhere(np.diff(np.sign(np.subtract(y1,y2))) != 0).reshape(-1) + 0

### Position of the intersections.
intersec = np.zeros((len(idx),2))
for i in range(len(idx)):
    x_pos = (vm[idx[i]]+vm[idx[i]+1])/2.
    y_pos = (y1[idx[i]]+y1[idx[i]+1])/2.
    intersec[i] = x_pos,y_pos


### Plot intersections

In [ ]:
fig, ax = plt.subplots(figsize=(15, 8), dpi= 80, facecolor='w', edgecolor='k')
ax.vlines(Vcut/mV, (intersec[1][0]-100), (intersec[1][-1]+100), lw=2, color='k')
plt.plot(vm,y1, label="Vm Nullcline")
plt.plot(vm,y2, label="W Nullcline")
for i in range(len(idx)):
    plt.plot(intersec[i][0],intersec[i][1], 'ro')
xt = ax.get_xticks() 
xt=np.append(xt,Vcut/mV)
xtl=xt.tolist()
xtl[-1]="$V_{threshold}$"
ax.set_xticks(xt)
ax.set_xticklabels(xtl)
xlabel('$v_m (mV)$')
ylabel('$w (nA)$')
plt.xlim([(intersec[0][0]-20), (intersec[1][0]+20)])
plt.ylim([(intersec[0][1]-100), (intersec[1][-1]+100)])
legend();
plt.show() 


## Verify your nullclines

In [ ]:
### Equations of dvm/dt and dw/dt
fvm = vm_0-w1
fw = w_0-w1

### Intersection point 1
f1 = fvm.subs(vm1,intersec[0][0])
f1 = f1.subs(w1,intersec[0][1])
print('\ndvm/dt = \n',round(f1,1))

### Intersection point 2
f2 = fw.subs(vm1,intersec[0][0])
f2 = f2.subs(w1,intersec[0][1])
print('\ndw/dt = \n',round(f2,1))


## Define the ODEs to plot the streamlines

In [ ]:
def f(Y, t, C,gL,taum,EL,VT,DeltaT,Vcut,tauw,a,b,I):
    C=C/pfarad # Can be fixed
    gL=gL/nsiemens
    taum=taum/msecond
    EL=EL/mvolt # Same as changing I
    VT=VT/mvolt
    DeltaT=DeltaT/mvolt
    Vcut=Vcut/mvolt
    I_input = I/nA
    tauw = tauw/msecond
    a = a/nsiemens
    b = b/nA
    vm, w = Y
    return [(gL*(EL-vm)+gL*DeltaT*exp((vm-VT)/DeltaT)+I_input-w)/C, (a*(vm-EL)-w)/tauw]

### Choose range for vm and w   
vm_stream = np.linspace(int((intersec[0][0]-20)), int((intersec[1][0]+20)), int((2*(intersec[1][0]-intersec[0][0]))))
w_stream = np.linspace(int((intersec[0][1]-500)), int((intersec[1][1]+200)), int((2*(intersec[1][1]-intersec[0][1]))))

### Colve the ODE
Y1, Y2 = np.meshgrid(vm_stream, w_stream)
t = 0
u, v = np.zeros(Y1.shape), np.zeros(Y2.shape)
NI, NJ = Y1.shape
for i in range(NI):
    for j in range(NJ):
        x = Y1[i, j]
        y = Y2[i, j]
        yprime = f([x, y], t, C,gL,taum,EL,VT,DeltaT,Vcut,neuron.namespace['tauw'],neuron.namespace['a'],neuron.namespace['b'],1.*namp)
        u[i,j] = yprime[0]
        v[i,j] = yprime[1]


## Plot nullclines with streamlines

In [ ]:
### Get a list with the solution to the equations. Substitute values of vm in the equations vm_0 and w_0. 
# This time use stream values.
y1_stream = []
for i in range(len(vm_stream)):
  y1_stream.append(vm_0.subs(vm1,vm_stream[i]))
y2_stream = []
for i in range(len(vm_stream)):
  y2_stream.append(w_0.subs(vm1,vm_stream[i]))

fig, ax = plt.subplots(figsize=(20, 10), dpi= 80, facecolor='w', edgecolor='k')
Q = ax.streamplot(Y1, Y2, u, v)
ax.vlines(Vcut/mV, (intersec[1][0]-200), (intersec[1][-1]+200), lw=2, color='k')
plt.plot(vm_stream,y1_stream, label="Vm Nullcline")
plt.plot(vm_stream,y2_stream, label="W Nullcline")
xt = ax.get_xticks() 
xt=np.append(xt,Vcut/mV)
xtl=xt.tolist()
xtl[-1]="$V_{threshold}$"
ax.set_xticks(xt)
ax.set_xticklabels(xtl)
xlabel('$v_m (mV)$')
ylabel('$w (nA)$')
plt.xlim([(intersec[0][0]-20), (intersec[1][0]+20)])
plt.ylim([(int(min(y1_stream))-30), (intersec[1][-1]+200)])
legend();
plt.show()


## Plot nullclines, streamlines and orbit of spiking neuron

In [ ]:
vm_orbit = mon.vm[0]/mV
w_orbit = mon.w[0]/nA

fig, ax = plt.subplots(figsize=(20, 10), dpi= 80, facecolor='w', edgecolor='k')
Q = ax.streamplot(Y1, Y2, u, v)
ax.vlines(Vcut/mV, (intersec[1][0]-200), (intersec[1][-1]+200), lw=2, color='k')
plt.plot(vm_orbit,w_orbit, label="Orbit", lw=4, color='r')
plt.plot(vm_stream,y1_stream, label="Vm Nullcline")
plt.plot(vm_stream,y2_stream, label="W Nullcline")
xt = ax.get_xticks() 
xt=np.append(xt,Vcut/mV)
xtl=xt.tolist()
xtl[-1]="$V_{threshold}$"
ax.set_xticks(xt)
ax.set_xticklabels(xtl)
xlabel('$v_m (mV)$')
ylabel('$w (nA)$')
plt.xlim([(intersec[0][0]-20), (intersec[1][0]+20)])
plt.ylim([(int(min(y1_stream))-30), (intersec[1][-1]+200)])
legend();
plt.show()


## Compute Jacobian Matrix to determine type of the Equilibria

The Jacobian matrix at the fixed point equilibria give the stability of the fixed points.

Let's consider the function **f**. You may recognize the AdEx equations. 

\begin{align*}
\textbf{f}(v_m,w) =& 
\begin{bmatrix} 
\frac{1}{C} \left(\Delta_T\cdot g_L \cdot e^{\frac{1}{\Delta_T} \left(- V_T + vm\right)} + I_{input} + g_L \left(E_L - v_m\right) - w\right) \\ 
\frac{1}{tau_w} \left(a \left(- E_L + v_m\right) - w\right)
\end{bmatrix}\\
\end{align*}

Then we have 
\begin{align*}
f_1(v_m,w) =& \frac{1}{C} \left(\Delta_T\cdot g_L\cdot e^{\frac{1}{\Delta_T} \left(- V_T + v_m\right)} + I_{input} + g_L \left(E_L - v_m\right) - w\right)\\
\end{align*}

and
\begin{align*}
f_2(v_m,w) =& \frac{1}{tau_w} \left(a \left(- E_L + v_m\right) - w\right)\\
\end{align*}

The Jacobian matrix of **f** is:
\begin{align*}
\textbf{J}_{\textbf{f}}(v_m,w) =& \begin{bmatrix} 
\frac{\partial f_1}{\partial v_m} & \frac{\partial }{\partial w}\\
\frac{\partial f_2}{\partial v_m} & \frac{\partial f_2}{\partial w}
\end{bmatrix}
 = \begin{bmatrix} 
\frac{\partial \left(\frac{1}{C} \left(\Delta_T\cdot g_L \cdot e^{\frac{1}{\Delta_T} \left(- V_T + v_m\right)} + I_{input} + g_L \left(E_L - v_m\right) - w\right)\right)}{\partial v_m} & \frac{\partial \left(\frac{1}{C} \left(\Delta_T\cdot g_L \cdot e^{\frac{1}{\Delta_T} \left(- V_T + vm\right)} + I_{input} + g_L \left(E_L - v_m\right) - w\right)\right)}{\partial w}\\
\frac{\partial \left(\frac{1}{tau_w} \left(a \left(- E_L + v_m\right) - w\right)\right)}{\partial v_m} & \frac{\partial \left(\frac{1}{tau_w} \left(a \left(- E_L + v_m\right) - w\right)\right)}{\partial w}
\end{bmatrix}\\
\end{align*}

### Install Symengine

In [ ]:
pip install symengine


## State Function for the Nullclines Symbolically 

In [ ]:
### Write equations in symbol format
fvm = vm_0 - w1
fw = w_0 - w1

### Find the function f
print('\n f(vm,w) = \n', np.matrix([fvm,fw]))
print('\n f1(vm,w) = \n', fvm)
print('\n f2(vm,w) = \n', fw)


## Compute the Jacobian

In [ ]:
def Jacobian(v_str, f_list):
    vars = sp.symbols(v_str)
    f = sp.sympify(f_list)
    J = sp.zeros(len(f),len(vars))
    for i, fi in enumerate(f):
        for j, s in enumerate(vars):
            J[i,j] = sp.diff(fi, s)
    return J

J = Jacobian('vm w', [fvm,fw])

print('\nJacobian Matrix:\n',J)


Take the 2x2 matrix of first derivatives at each fixed point (the Jacobian) and compute its eigenvalues:


*   Two negative eigenvalues at a fix point implies that the fix point is stable (trajectories starting from neighboring points converge to it).
*   Two positive eigenvalues indicates an unstable fix point (trajectories starting from neighboring points diverge from it).
*   One eigenvalue of each sign corresponds to a saddle point (trajectories from some neighboring points converge to it and others diverge).


## Eigenvalue at each point




In [ ]:
## Point 1
J1 = J.subs(vm1,intersec[0][0])
JJ1 = np.array([[float(J1[0]),float(J1[1])],[float(J1[2]),float(J1[3])]])
e1 = np.linalg.eigvals(JJ1)

print('\nEigenvalues point 1:\n',e1)

## Point 2
J2 = J.subs(vm1,intersec[1][0])
JJ2 = np.array([[float(J2[0]),float(J2[1])],[float(J2[2]),float(J2[3])]])
e2 = np.linalg.eigvals(JJ2)

print('\nEigenvalues point 2:\n',e2)


### What type of fix points do we have?


*   For point 1:
*   For point 2:



#License

<a rel="license" href="http://creativecommons.org/licenses/by/4.0/"><img alt="Creative Commons License" style="border-width:0" src="https://i.creativecommons.org/l/by/4.0/88x31.png" /></a><br />This work is licensed under a <a rel="license" href="http://creativecommons.org/licenses/by/4.0/">Creative Commons Attribution 4.0 International License</a>.

Mario Negrello, Elias Santoro. 
